In [ ]:
# !ls -lh

from pathlib import Path

for path in sorted(Path.cwd().iterdir()):
    print(path.name)

total 9.8M
-rw------- 1 root root 892K May 18 18:14 niletts_llm_qa_test_reference.csv
-rw------- 1 root root 7.5M May 18 18:13 niletts_llm_qa_train.csv
-rw------- 1 root root 813K May 18 18:13 niletts_llm_qa_val.csv
-rw------- 1 root root 671K May 18 18:13 niletts_test_asr.csv


In [ ]:
# !nvidia-smi

import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi is not available in this environment.")

Mon May 18 18:14:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1C.0 Off |                    0 |
|  0%   35C    P8             16W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Fine-tune AraT5 for NileTTS QA + Evaluate Reference vs ASR Contexts

This notebook uses the final prepared CSV files:

- `niletts_llm_qa_train.csv`
- `niletts_llm_qa_val.csv`
- `niletts_llm_qa_test_reference.csv`
- `niletts_test_asr.csv`

Pipeline:

1. Load and validate QA train/val/test files.
2. Merge test QA pairs with ASR eval transcripts by `audio_file`.
3. Fine-tune `UBC-NLP/AraT5-base` on reference-transcript QA pairs.
4. Evaluate on reference transcripts.
5. Evaluate the same questions on ASR transcripts.
6. Compare EM/F1 drop caused by ASR errors.

Important:
- QA pairs were generated from `context_ref` / clean reference transcripts.
- ASR evaluation uses the same questions and same gold answers, but changes the context from `context_ref` to `context_asr`.


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-qa.txt").is_file():
        requirements_file = candidate / "requirements-qa.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-qa.txt")

%pip install -r "{requirements_file}"
# =====


In [4]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))


Device: cuda
GPU: NVIDIA A10G
CUDA capability: (8, 6)


## 1. File paths

Upload the four CSV files to the notebook environment, then set these paths if needed.


In [ ]:

# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
        and (candidate / "3_FINE_TUNED_QA_MODEL").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

QA_DIR = PROJECT_ROOT / "2_QA_GENERATION"
MODEL_DIR = PROJECT_ROOT / "3_FINE_TUNED_QA_MODEL"

# TRAIN_FILE = "niletts_llm_qa_train.csv"
# VAL_FILE = "niletts_llm_qa_val.csv"
# TEST_FILE = "niletts_llm_qa_test_reference.csv"
# ASR_FILE = "niletts_test_asr.csv"

TRAIN_FILE = (QA_DIR / "1_train_data" / "niletts_llm_qa_train.csv")
VAL_FILE = (QA_DIR / "2_val_data" / "niletts_llm_qa_val.csv")
TEST_FILE = (QA_DIR / "3_test_data"/ "niletts_llm_qa_test_reference.csv")
ASR_FILE = (PROJECT_ROOT / "1_ASR"/ "niletts_asr_eval_finetuning_transcriptions_metrics.csv")

ARTIFACT_DIR = MODEL_DIR / "1_arat5_niletts_final_outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_TEST_FILE = (
    ARTIFACT_DIR
    / "niletts_qa_test_reference_with_asr_context.csv"
)

for path in [TRAIN_FILE, VAL_FILE, TEST_FILE, ASR_FILE]:
    # if not Path(path).exists():
    if not path.is_file():
        # raise FileNotFoundError(f"Missing file: {path}. Upload it or update the path.")
        raise FileNotFoundError(f"Missing file: {path}")

# print("All files found.")
print("All input files found.")
# =====


All files found.


## 2. Load and inspect data

In [6]:
qa_train_df = pd.read_csv(TRAIN_FILE)
qa_val_df = pd.read_csv(VAL_FILE)
qa_test_df = pd.read_csv(TEST_FILE)
asr_df = pd.read_csv(ASR_FILE)

print("Train:", qa_train_df.shape)
print("Val:", qa_val_df.shape)
print("Test QA:", qa_test_df.shape)
print("ASR eval:", asr_df.shape)

display(qa_train_df.head(2))
display(qa_val_df.head(2))
display(qa_test_df.head(2))
display(asr_df.head(2))


Train: (12877, 18)
Val: (1398, 16)
Test QA: (1554, 16)
ASR eval: (950, 6)


,id,split,part_id,metadata_row_index,qa_train_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,difficulty,needs_manual_review,question_type_raw,difficulty_raw
0,train_6682_0,train,3,101,6682,wavs/sales_11_00000.wav,SPEAKER_02,sales,أهلاً بيكم معانا. النهاردة هنتكلم في موضوع مهم...,الشراكة هي سلاح زو حدين؟,سلاح زو حدين,236,True,domain_specific,easy,True,domain_specific,easy
1,train_6220_0,train,3,6542,6220,wavs/sales_33_00138.wav,SPEAKER_01,sales,المدينة كلها على بعضها كده رواية معمارية بتحكي...,اللغة التي ذكرت في بداية السياق هي:,أحسن نقدر قيمة الأماكن اللي بنعيش فيها أكتر.,131,True,domain_specific,easy,True,domain_specific,easy


,id,split,qa_val_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review
0,val_1_0,val,1,wavs/sales_22_00010.wav,SPEAKER_02,sales,للشركة بتصرفاته هو شخصياً. لو هو هادي و بيحترم...,الشركة تتعامل مع العملاء بشكل شخصي؟,لو هو هادي و بيحترم الناس و بيسمع الفريق بيقلده.,27,True,domain_specific,domain_specific,easy,easy,True
1,val_1_1,val,1,wavs/sales_22_00010.wav,SPEAKER_02,sales,للشركة بتصرفاته هو شخصياً. لو هو هادي و بيحترم...,كيف يمكن أن يتعلم الفريق من الشخص العصبي؟,لو هو عصبي و بيلوم على طول الفريق بيتعلم يخاف ...,76,True,detail_extraction,detail_extraction,medium,medium,True


,id,split,qa_test_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review
0,test_0_0,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,1,True,domain_specific,domain_specific,easy,easy,True
1,test_0_1,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,105,True,intent,intent/main_idea,medium,medium,True


,audio_file,speaker_name,reference_text,asr_text,wer,cer
0,wavs/sales_34_00019.wav,SPEAKER_02,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407
1,wavs/sales_26_00082.wav,SPEAKER_01,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000


## 3. Validate required columns and clean text

In [7]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def validate_qa_df(df, name):
    required = [
        "audio_file", "speaker_name", "domain", "context_ref",
        "question", "answer", "valid_answer_span_ref"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

    out = df.copy()

    for col in ["audio_file", "speaker_name", "domain", "context_ref", "question", "answer"]:
        out[col] = out[col].astype(str).map(normalize_spaces)

    out["answer_start_ref"] = out.apply(
        lambda r: find_answer_start_soft(r["context_ref"], r["answer"]),
        axis=1
    )

    out["valid_answer_span_ref"] = out["answer_start_ref"] != -1

    before = len(out)
    out = out[out["question"].str.len() > 5]
    out = out[out["answer"].str.len() > 0]
    out = out[out["valid_answer_span_ref"] == True]
    out = out.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

    print(f"{name}: kept {len(out)} / {before}")
    print(f"{name}: unique audio files = {out['audio_file'].nunique()}")

    return out


qa_train_df = validate_qa_df(qa_train_df, "train")
qa_val_df = validate_qa_df(qa_val_df, "val")
qa_test_df = validate_qa_df(qa_test_df, "test")

required_asr = ["audio_file", "speaker_name", "reference_text", "asr_text", "wer", "cer"]
missing_asr = [c for c in required_asr if c not in asr_df.columns]
if missing_asr:
    raise ValueError(f"ASR file missing columns: {missing_asr}")

asr_df = asr_df.copy()
for col in ["audio_file", "speaker_name", "reference_text", "asr_text"]:
    asr_df[col] = asr_df[col].astype(str).map(normalize_spaces)

asr_df["wer"] = pd.to_numeric(asr_df["wer"], errors="coerce")
asr_df["cer"] = pd.to_numeric(asr_df["cer"], errors="coerce")

print("ASR WER:", asr_df["wer"].mean())
print("ASR CER:", asr_df["cer"].mean())


train: kept 12877 / 12877
train: unique audio files = 5962
val: kept 1398 / 1398
val: unique audio files = 665
test: kept 1554 / 1554
test: unique audio files = 738
ASR WER: 0.11683900680229063
ASR CER: 0.038978167019728595


## 4. Data quality checks

These checks help prove there is no split leakage and the ASR file matches the test QA file.


In [8]:
def show_split_stats(name, df, source_col=None):
    print("\n" + "="*40)
    print(name)
    print("Rows:", len(df))
    print("Unique audio files:", df["audio_file"].nunique())
    if source_col and source_col in df.columns:
        print(f"Unique {source_col}:", df[source_col].nunique())
    print("Domain distribution:")
    display(df["domain"].value_counts())
    if "question_type" in df.columns:
        print("Question type distribution:")
        display(df["question_type"].value_counts(dropna=False))
    if "difficulty" in df.columns:
        print("Difficulty distribution:")
        display(df["difficulty"].value_counts(dropna=False))


show_split_stats("Train", qa_train_df, "qa_train_source_index")
show_split_stats("Val", qa_val_df, "qa_val_source_index")
show_split_stats("Test", qa_test_df, "qa_test_source_index")

train_audio = set(qa_train_df["audio_file"])
val_audio = set(qa_val_df["audio_file"])
test_audio = set(qa_test_df["audio_file"])

print("\nSplit overlap checks:")
print("Train ∩ Val:", len(train_audio & val_audio))
print("Train ∩ Test:", len(train_audio & test_audio))
print("Val ∩ Test:", len(val_audio & test_audio))

asr_audio = set(asr_df["audio_file"])
print("\nASR/test matching:")
print("Test QA audio missing in ASR:", len(test_audio - asr_audio))
print("ASR audio not used by filtered QA test:", len(asr_audio - test_audio))



Train
Rows: 12877
Unique audio files: 5962
Unique qa_train_source_index: 5962
Domain distribution:


domain
sales      6984
general    4009
medical    1884
Name: count, dtype: int64

Question type distribution:


question_type
detail_extraction    4621
domain_specific      4503
intent               2322
main_idea            1168
number_extraction     263
Name: count, dtype: int64

Difficulty distribution:


...

Question type distribution:


question_type
detail_extraction    551
domain_specific      492
intent               322
main_idea            162
number_extraction     27
Name: count, dtype: int64

Difficulty distribution:


difficulty
easy      1044
medium     510
Name: count, dtype: int64


Split overlap checks:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

ASR/test matching:
Test QA audio missing in ASR: 0
ASR audio not used by filtered QA test: 212


## 5. Merge test QA with ASR transcripts

Do not filter test rows based on whether the answer appears in `context_asr`.

If the answer does not appear in ASR transcript, that is part of the ASR-error effect we want to measure.


In [ ]:
qa_test_with_asr_df = qa_test_df.merge(
    asr_df[["audio_file", "asr_text", "wer", "cer", "reference_text"]],
    on="audio_file",
    how="left"
).rename(columns={
    "asr_text": "context_asr",
    "wer": "asr_wer",
    "cer": "asr_cer"
})

missing_asr_context = qa_test_with_asr_df["context_asr"].isna().sum()
print("Missing ASR contexts after merge:", missing_asr_context)

if missing_asr_context > 0:
    raise ValueError("Some test QA rows do not have ASR transcripts. Check audio_file matching.")

qa_test_with_asr_df["answer_in_asr_exact"] = qa_test_with_asr_df.apply(
    lambda r: str(r["answer"]) in str(r["context_asr"]),
    axis=1
)

print("Test QA rows:", len(qa_test_with_asr_df))
print("Unique test audios:", qa_test_with_asr_df["audio_file"].nunique())
print("Answer appears exactly inside ASR context:", qa_test_with_asr_df["answer_in_asr_exact"].mean())
print("Mean ASR WER over QA rows:", qa_test_with_asr_df["asr_wer"].mean())
print("Mean ASR CER over QA rows:", qa_test_with_asr_df["asr_cer"].mean())

# =====
# qa_test_with_asr_df.to_csv(
#     "niletts_qa_test_reference_with_asr_context.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

qa_test_with_asr_df.to_csv(
    MERGED_TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)
# =====

display(qa_test_with_asr_df.head(2))


Missing ASR contexts after merge: 0
Test QA rows: 1554
Unique test audios: 738
Answer appears exactly inside ASR context: 0.5096525096525096
Mean ASR WER over QA rows: 0.11756393013612884
Mean ASR CER over QA rows: 0.04121027254249641


,id,split,qa_test_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,...,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review,context_asr,asr_wer,asr_cer,reference_text,answer_in_asr_exact
0,test_0_0,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,1,...,domain_specific,domain_specific,easy,easy,True,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,False
1,test_0_1,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,105,...,intent,intent/main_idea,medium,medium,True,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,True


## 6. Prepare AraT5 input format

The model input is:

```text
أجب عن السؤال اعتمادا على النص. السؤال: ... النص: ...
```

The target is the gold answer.


In [10]:
MODEL_NAME = "UBC-NLP/AraT5-base"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 96

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Gradient checkpointing saves GPU memory. It can make training slightly slower.
USE_GRADIENT_CHECKPOINTING = True

if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

model.to(device)

print("Loaded:", MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loaded: UBC-NLP/AraT5-base


In [11]:
def build_t5_input(question, context):
    return f"أجب عن السؤال اعتمادا على النص. السؤال: {question} النص: {context}"


def prepare_for_t5(df, context_col="context_ref"):
    out = df.copy()
    out["input_text"] = out.apply(
        lambda r: build_t5_input(r["question"], r[context_col]),
        axis=1
    )
    out["target_text"] = out["answer"].astype(str)
    return out


t5_train_df = prepare_for_t5(qa_train_df, context_col="context_ref")
t5_val_df = prepare_for_t5(qa_val_df, context_col="context_ref")
t5_test_ref_df = prepare_for_t5(qa_test_with_asr_df, context_col="context_ref")
t5_test_asr_df = prepare_for_t5(qa_test_with_asr_df, context_col="context_asr")

train_dataset = Dataset.from_pandas(t5_train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(t5_val_df[["input_text", "target_text"]])
test_ref_dataset = Dataset.from_pandas(t5_test_ref_df[["input_text", "target_text"]])
test_asr_dataset = Dataset.from_pandas(t5_test_asr_df[["input_text", "target_text"]])

print(train_dataset)
print(val_dataset)
print(test_ref_dataset)
print(test_asr_dataset)


Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 12877
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 1398
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 1554
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 1554
})


In [12]:
def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test_ref = test_ref_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=test_ref_dataset.column_names
)

tokenized_test_asr = test_asr_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=test_asr_dataset.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


Map:   0%|          | 0/12877 [00:00<?, ? examples/s]

Map:   0%|          | 0/1398 [00:00<?, ? examples/s]

Map:   0%|          | 0/1554 [00:00<?, ? examples/s]

Map:   0%|          | 0/1554 [00:00<?, ? examples/s]

## 7. Metrics: Arabic-normalized Exact Match and token F1

In [19]:
AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
PUNCT = re.compile(r"[^\w\s\u0600-\u06FF]")


def normalize_arabic(text):
    text = str(text)
    text = AR_DIACRITICS.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = PUNCT.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match_score(prediction, ground_truth):
    return int(normalize_arabic(prediction) == normalize_arabic(ground_truth))


def token_f1_score(prediction, ground_truth):
    pred_tokens = normalize_arabic(prediction).split()
    gold_tokens = normalize_arabic(ground_truth).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = {}
    for tok in set(pred_tokens):
        common[tok] = min(pred_tokens.count(tok), gold_tokens.count(tok))

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def clean_token_ids(token_ids):
    """
    Makes prediction/label IDs safe before tokenizer.batch_decode().
    Handles:
    - tensors
    - logits instead of token IDs
    - -100 padding
    - out-of-vocabulary IDs
    """
    if isinstance(token_ids, tuple):
        token_ids = token_ids[0]

    if torch.is_tensor(token_ids):
        token_ids = token_ids.detach().cpu().numpy()

    token_ids = np.asarray(token_ids)

    # If predictions are logits: shape = [batch, seq_len, vocab]
    if token_ids.ndim == 3:
        token_ids = np.argmax(token_ids, axis=-1)

    token_ids = token_ids.astype(np.int64)

    vocab_size = len(tokenizer)

    token_ids = np.where(token_ids < 0, tokenizer.pad_token_id, token_ids)
    token_ids = np.where(token_ids >= vocab_size, tokenizer.pad_token_id, token_ids)

    return token_ids


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    preds = clean_token_ids(preds)
    labels = clean_token_ids(labels)

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    em_scores = [
        exact_match_score(p, g)
        for p, g in zip(decoded_preds, decoded_labels)
    ]

    f1_scores = [
        token_f1_score(p, g)
        for p, g in zip(decoded_preds, decoded_labels)
    ]

    return {
        "exact_match": float(np.mean(em_scores)),
        "f1": float(np.mean(f1_scores))
    }


## 8. Training arguments with checkpoints

The trainer saves checkpoints every `SAVE_EVERY_STEPS`.

If the notebook stops, rerun from the top and the training cell will resume from the last checkpoint automatically.


In [ ]:
# =====
# OUTPUT_DIR = "arat5_niletts_llm_reference_qa"
# BEST_MODEL_DIR = "arat5_niletts_llm_reference_qa_best"

OUTPUT_DIR = str(
    MODEL_DIR / "arat5_niletts_llm_reference_qa"
)

BEST_MODEL_DIR = str(
    ARTIFACT_DIR / "arat5_niletts_llm_reference_qa_best"
)
# =====

# Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
SAVE_EVERY_STEPS = 500
EVAL_EVERY_STEPS = 500
LOGGING_STEPS = 50

# Use bf16 on Ampere/Hopper GPUs if available; otherwise use fp16.
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    USE_BF16 = major >= 8
    USE_FP16 = not USE_BF16
else:
    USE_BF16 = False
    USE_FP16 = False

print("USE_BF16:", USE_BF16)
print("USE_FP16:", USE_FP16)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)


USE_BF16: True
USE_FP16: False
Effective train batch size: 8


In [24]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,

    fp16=USE_FP16,
    bf16=USE_BF16,

    # IMPORTANT FIX: Avoid safetensors non-contiguous tensor saving error.
    save_safetensors=False,

    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## 9. Train / resume training

If a checkpoint exists inside `OUTPUT_DIR`, this cell resumes automatically.


In [25]:
last_checkpoint = None

if Path(OUTPUT_DIR).exists():
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

print("Last checkpoint:", last_checkpoint)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print("Saved best model to:", BEST_MODEL_DIR)


Last checkpoint: None


Step,Training Loss,Validation Loss,Exact Match,F1
500,6.445500,6.556940,0.000000,0.016742
1000,6.529000,5.594008,0.000715,0.028387
1500,5.652800,4.637224,0.001431,0.062171
2000,5.163300,4.029428,0.002146,0.097117
2500,4.763700,3.618241,0.002146,0.113728
3000,4.391100,3.362248,0.002861,0.131687
3500,4.281800,3.166741,0.004292,0.141610
4000,4.030100,3.050777,0.005007,0.148382
4500,4.020100,2.967132,0.005722,0.153023


***** train metrics *****
  epoch                    =     2.9986
  total_flos               =  3075773GF
  train_loss               =     5.1011
  train_runtime            = 2:24:20.65
  train_samples_per_second =      4.461
  train_steps_per_second   =      0.557
Saved best model to: arat5_niletts_llm_reference_qa_best


In [23]:
# import shutil
# from pathlib import Path

# OUTPUT_DIR = "arat5_niletts_llm_reference_qa"

# output_path = Path(OUTPUT_DIR)

# if output_path.exists():
#     for ckpt in output_path.glob("checkpoint-*"):
#         has_state = (ckpt / "trainer_state.json").exists()
#         has_model = (ckpt / "pytorch_model.bin").exists() or (ckpt / "model.safetensors").exists()

#         if not has_state or not has_model:
#             print("Deleting incomplete checkpoint:", ckpt)
#             shutil.rmtree(ckpt)

# print("Remaining checkpoints:")
# if output_path.exists():
#     for ckpt in sorted(output_path.glob("checkpoint-*")):
#         print(ckpt)

Deleting incomplete checkpoint: arat5_niletts_llm_reference_qa/checkpoint-500
Remaining checkpoints:


## 10. Evaluate on reference transcript contexts

In [ ]:
ref_pred_output = trainer.predict(tokenized_test_ref)

ref_pred_ids = clean_token_ids(ref_pred_output.predictions)
ref_label_ids = clean_token_ids(ref_pred_output.label_ids)

ref_pred_texts = tokenizer.batch_decode(
    ref_pred_ids,
    skip_special_tokens=True
)

gold_texts = tokenizer.batch_decode(
    ref_label_ids,
    skip_special_tokens=True
)

eval_ref_df = qa_test_with_asr_df.copy()
eval_ref_df["gold_answer"] = gold_texts
eval_ref_df["pred_answer_ref_context"] = ref_pred_texts

eval_ref_df["em_ref"] = [
    exact_match_score(p, g)
    for p, g in zip(ref_pred_texts, gold_texts)
]

eval_ref_df["f1_ref"] = [
    token_f1_score(p, g)
    for p, g in zip(ref_pred_texts, gold_texts)
]

ref_em = eval_ref_df["em_ref"].mean()
ref_f1 = eval_ref_df["f1_ref"].mean()

print("Reference Transcript QA Exact Match:", ref_em)
print("Reference Transcript QA F1:", ref_f1)

# =====
# eval_ref_df.to_csv(
#     "niletts_arat5_llm_reference_eval_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

eval_ref_df.to_csv(
    ARTIFACT_DIR / "niletts_arat5_llm_reference_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(eval_ref_df[[
    "audio_file", "domain", "question", "context_ref",
    "gold_answer", "pred_answer_ref_context", "em_ref", "f1_ref"
]].head())


Reference Transcript QA Exact Match: 0.005148005148005148
Reference Transcript QA F1: 0.15471839560643455


,audio_file,domain,question,context_ref,gold_answer,pred_answer_ref_context,em_ref,f1_ref
0,wavs/sales_34_00019.wav,sales,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص اللي هو يعني حافز مش فاهم ده.,مش فاهم ده.,0,0.545455
1,wavs/sales_34_00019.wav,sales,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الأهم دلوقتي الشخص اللي عنده فضول.,مش فاهم ده.,0,0.000000
2,wavs/sales_34_00019.wav,sales,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ده ممكن يبقى مشكلة بعدين.,مش فاهم ده.,0,0.250000
3,wavs/sales_26_00082.wav,sales,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,بناء فريق عمل ناجح.,مش يقل أهمية أبدا.,0,0.000000
4,wavs/sales_26_00082.wav,sales,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,موضوع تاني لا يقل أهمية أبدا.,مش يقل أهمية أبدا.,0,0.600000


## 11. Evaluate on ASR transcript contexts

Same QA questions, same gold answers, but context is `context_asr`.


In [ ]:
asr_pred_output = trainer.predict(tokenized_test_asr)

# IMPORTANT: clean token IDs before decoding
asr_pred_ids = clean_token_ids(asr_pred_output.predictions)
asr_label_ids = clean_token_ids(asr_pred_output.label_ids)

asr_pred_texts = tokenizer.batch_decode(
    asr_pred_ids,
    skip_special_tokens=True
)

asr_gold_texts = tokenizer.batch_decode(
    asr_label_ids,
    skip_special_tokens=True
)

eval_asr_df = qa_test_with_asr_df.copy()
eval_asr_df["gold_answer"] = asr_gold_texts
eval_asr_df["pred_answer_asr_context"] = asr_pred_texts

eval_asr_df["em_asr"] = [
    exact_match_score(p, g)
    for p, g in zip(asr_pred_texts, asr_gold_texts)
]

eval_asr_df["f1_asr"] = [
    token_f1_score(p, g)
    for p, g in zip(asr_pred_texts, asr_gold_texts)
]

asr_em = eval_asr_df["em_asr"].mean()
asr_f1 = eval_asr_df["f1_asr"].mean()

print("ASR Transcript QA Exact Match:", asr_em)
print("ASR Transcript QA F1:", asr_f1)

# =====
# eval_asr_df.to_csv(
#     "niletts_arat5_llm_asr_eval_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

eval_asr_df.to_csv(
    ARTIFACT_DIR / "niletts_arat5_llm_asr_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(eval_asr_df[[
    "audio_file", "domain", "asr_wer", "asr_cer", "question", "context_asr",
    "gold_answer", "pred_answer_asr_context", "em_asr", "f1_asr"
]].head())

ASR Transcript QA Exact Match: 0.002574002574002574
ASR Transcript QA F1: 0.14694174939088134


,audio_file,domain,asr_wer,asr_cer,question,context_asr,gold_answer,pred_answer_asr_context,em_asr,f1_asr
0,wavs/sales_34_00019.wav,sales,0.035714,0.007407,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,الشخص اللي هو يعني حافز مش فاهم ده.,مش فاهم ده.,0,0.545455
1,wavs/sales_34_00019.wav,sales,0.035714,0.007407,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,الأهم دلوقتي الشخص اللي عنده فضول.,مش فاهم ده.,0,0.000000
2,wavs/sales_34_00019.wav,sales,0.035714,0.007407,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,ده ممكن يبقى مشكلة بعدين.,مش فاهم ده.,0,0.250000
3,wavs/sales_26_00082.wav,sales,0.000000,0.000000,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,بناء فريق عمل ناجح.,مش يعل أهمية أبدا.,0,0.000000
4,wavs/sales_26_00082.wav,sales,0.000000,0.000000,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,موضوع تاني لا يقل أهمية أبدا.,مش يعل أهمية أبدا.,0,0.400000


## 12. Compare reference-vs-ASR QA drop

In [ ]:
comparison_df = eval_ref_df[[
    "audio_file",
    "domain",
    "question",
    "answer",
    "context_ref",
    "context_asr",
    "asr_wer",
    "asr_cer",
    "answer_in_asr_exact",
    "pred_answer_ref_context",
    "em_ref",
    "f1_ref",
]].copy()

comparison_df["pred_answer_asr_context"] = eval_asr_df["pred_answer_asr_context"].values
comparison_df["em_asr"] = eval_asr_df["em_asr"].values
comparison_df["f1_asr"] = eval_asr_df["f1_asr"].values

comparison_df["em_drop"] = comparison_df["em_ref"] - comparison_df["em_asr"]
comparison_df["f1_drop"] = comparison_df["f1_ref"] - comparison_df["f1_asr"]

summary_metrics = {
    "reference_em": float(comparison_df["em_ref"].mean()),
    "reference_f1": float(comparison_df["f1_ref"].mean()),
    "asr_em": float(comparison_df["em_asr"].mean()),
    "asr_f1": float(comparison_df["f1_asr"].mean()),
    "em_drop": float(comparison_df["em_drop"].mean()),
    "f1_drop": float(comparison_df["f1_drop"].mean()),
    "asr_wer_mean": float(comparison_df["asr_wer"].mean()),
    "asr_cer_mean": float(comparison_df["asr_cer"].mean()),
    "answer_in_asr_exact_rate": float(comparison_df["answer_in_asr_exact"].mean()),
    "num_test_qa_pairs": int(len(comparison_df)),
    "num_test_audio_files": int(comparison_df["audio_file"].nunique()),
}

print(json.dumps(summary_metrics, ensure_ascii=False, indent=2))

# =====
# with open("niletts_arat5_reference_vs_asr_summary_metrics.json", "w", encoding="utf-8") as f:
with open(ARTIFACT_DIR / "niletts_arat5_reference_vs_asr_summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, ensure_ascii=False, indent=2)
# =====

# =====
# comparison_df.to_csv(
#     "niletts_arat5_reference_vs_asr_comparison.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

comparison_df.to_csv(
    ARTIFACT_DIR / "niletts_arat5_reference_vs_asr_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(comparison_df.head())


{
  "reference_em": 0.005148005148005148,
  "reference_f1": 0.15471839560643455,
  "asr_em": 0.002574002574002574,
  "asr_f1": 0.14694174939088134,
  "em_drop": 0.002574002574002574,
  "f1_drop": 0.007776646215553203,
  "asr_wer_mean": 0.11756393013612884,
  "asr_cer_mean": 0.04121027254249641,
  "answer_in_asr_exact_rate": 0.5096525096525096,
  "num_test_qa_pairs": 1554,
  "num_test_audio_files": 738
}


,audio_file,domain,question,answer,context_ref,context_asr,asr_wer,asr_cer,answer_in_asr_exact,pred_answer_ref_context,em_ref,f1_ref,pred_answer_asr_context,em_asr,f1_asr,em_drop,f1_drop
0,wavs/sales_34_00019.wav,sales,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,False,مش فاهم ده.,0,0.545455,مش فاهم ده.,0,0.545455,0,0.0
1,wavs/sales_34_00019.wav,sales,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,True,مش فاهم ده.,0,0.000000,مش فاهم ده.,0,0.000000,0,0.0
2,wavs/sales_34_00019.wav,sales,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,ده ممكن يبقى مشكلة بعدين.,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,فالشخص اللي هو يعني حافظ مش فاهم ده. اللي قافل...,0.035714,0.007407,True,مش فاهم ده.,0,0.250000,مش فاهم ده.,0,0.250000,0,0.0
3,wavs/sales_26_00082.wav,sales,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,بناء فريق عمل ناجح.,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000,True,مش يقل أهمية أبدا.,0,0.000000,مش يعل أهمية أبدا.,0,0.000000,0,0.0
4,wavs/sales_26_00082.wav,sales,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,موضوع تاني لا يقل أهمية أبداً.,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,0.000000,0.000000,True,مش يقل أهمية أبدا.,0,0.600000,مش يعل أهمية أبدا.,0,0.400000,0,0.2


## 13. Domain-level analysis

In [ ]:
domain_summary = comparison_df.groupby("domain").agg(
    qa_pairs=("audio_file", "count"),
    unique_audio=("audio_file", "nunique"),
    mean_asr_wer=("asr_wer", "mean"),
    mean_asr_cer=("asr_cer", "mean"),
    reference_em=("em_ref", "mean"),
    reference_f1=("f1_ref", "mean"),
    asr_em=("em_asr", "mean"),
    asr_f1=("f1_asr", "mean"),
    mean_f1_drop=("f1_drop", "mean"),
    answer_in_asr_exact_rate=("answer_in_asr_exact", "mean"),
).reset_index()

# =====
# domain_summary.to_csv(
#     "niletts_arat5_domain_level_results.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

domain_summary.to_csv(
    ARTIFACT_DIR / "niletts_arat5_domain_level_results.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(domain_summary)


,domain,qa_pairs,unique_audio,mean_asr_wer,mean_asr_cer,reference_em,reference_f1,asr_em,asr_f1,mean_f1_drop,answer_in_asr_exact_rate
0,general,495,233,0.097940,0.033252,0.000000,0.161213,0.000000,0.144024,0.017189,0.541414
1,medical,212,101,0.140479,0.075467,0.004717,0.145890,0.000000,0.147863,-0.001974,0.584906
2,sales,847,404,0.123297,0.037287,0.008264,0.153133,0.004723,0.148416,0.004716,0.472255


## 14. Save worst error examples

In [ ]:
worst_examples = comparison_df.sort_values(
    by=["f1_drop", "asr_wer"],
    ascending=[False, False]
).head(50)

# =====
# worst_examples.to_csv(
#     "niletts_arat5_worst_asr_error_examples.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

worst_examples.to_csv(
    ARTIFACT_DIR / "niletts_arat5_worst_asr_error_examples.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====


display(worst_examples[[
    "audio_file", "domain", "asr_wer", "question", "answer",
    "pred_answer_ref_context", "pred_answer_asr_context",
    "f1_ref", "f1_asr", "f1_drop",
    "context_ref", "context_asr"
]].head(10))


,audio_file,domain,asr_wer,question,answer,pred_answer_ref_context,pred_answer_asr_context,f1_ref,f1_asr,f1_drop,context_ref,context_asr
573,wavs/sales_19_00097.wav,sales,0.212121,السوق الحقيقي مع العملاء الحقيقيين؟,في السوق الحقيقي مع عملاء حقيقيين.,في السوق الحقيقي مع عملاء حقيقيين.,كل ممكن و جاهز دايما للتطوير و للتطوير و للتطو...,1.000000,0.000000,1.000000,في السوق الحقيقي مع عملاء حقيقيين. اسمع بانتبا...,في السوق الحقيقي مع عملاء حقيقيين. اسمع بانتبا...
902,wavs/sales_2_00148.wav,sales,0.157895,ما هي التكاليف المضافة عند الحصول على عائد أعل...,معاه مجهود و تكلفة إدارة أعلى بكتير.,لو ده معاه مجهود و تكلفة إدارة أعلى بكتير.,ممكن يوصل للأقل أو أكتر كمان.,0.875000,0.000000,0.875000,ممكن يجيب عائد أعلى بكتير من الإيجار السنوي ال...,ممكن يجيب عائد أعلى بكتير من الإيجار السنوى ال...
1149,wavs/sales_15_00041.wav,sales,0.483871,هل هذه هي واحدة من أفضل الأمثلة التي تم تقديمه...,هي دي أشهر مثال لها.,هو دي أشهر مثال لها.,مش بتاعت الخمس جنيه مش جنيه.,0.800000,0.000000,0.800000,آه بتاع التسعة و تسعين جنيه بدل مية. هي دي أشه...,آه بتاع 99 جنيه بدل 100. هي دي أشهر مثال لها 9...
647,wavs/sales_7_00093.wav,sales,0.000000,أي شيء يجعل هذا أفضل من شراء مساحة كبيرة في مو...,مش المساحة الأكبر أحسن؟ مش دايماً.,ممكن ده؟ مش المساحة الأكبر أحسن؟ مش دايما.,مش المساحة الأكبر أحسن؟ مش المساحة الأكبر أحسن...,0.857143,0.121951,0.735192,ده غالباً أحسن بكتير من إنك تشتري مساحة ضخمة ف...,ده غالباً أحسن بكتير من إنك تشتري مساحة ضخمة ف...
648,wavs/sales_7_00093.wav,sales,0.000000,كيف يمكن أن يكون هذا أفضل من شراء مساحة كبيرة ...,مش المساحة الأكبر أحسن؟ مش دايماً.,ممكن ده؟ مش المساحة الأكبر أحسن؟ مش دايما.,مش المساحة الأكبر أحسن؟ مش المساحة الأكبر أحسن...,0.857143,0.121951,0.735192,ده غالباً أحسن بكتير من إنك تشتري مساحة ضخمة ف...,ده غالباً أحسن بكتير من إنك تشتري مساحة ضخمة ف...
468,wavs/sales_30_00047.wav,sales,0.094340,كيف يمكن لشخص أن يبدأ صفحة جديدة بشكل صحي بعد ...,ازاي يبدأ صفحة جديدة بشكل صحي؟,مش يبدأ روح جديدة بشكل صحي؟,في كل قصص للنجاح اللي في المصادر اللي عملائه ه...,0.666667,0.000000,0.666667,اهم بكتير اوي من النجاح السهلة من اول مرة. و د...,أهم بكتير اوي من النجاح السهلة من أول مرة. و د...
621,wavs/sales_33_00013.wav,sales,0.048780,كيف تحدد قدرتكم على النجاح؟,هي اللي بتحدد بشكل كبير قدرتنا على أننا نقوم و...,مش بتحدد بشكل كبير القدرة على تكون تأجل و نحاو...,لو الواحد بيقول هو فشلت في المحاولة دي.,0.640000,0.000000,0.640000,لأن الطريقة اللي بنستقبل بيها الفشل ده. و اللي...,لإن الطريقة اللي بنستقبل بيها الفشل ده و اللي ...
1230,wavs/general_40_00039.wav,general,0.083333,كيف يمكننا تلخيص أهمية السكشن من خلال هذه الكل...,هو الجسر اللي بيربط بين النظرية المجردة اللي ب...,كل الجسر اللي بيربط بين الفكرة النصية اللي بتت...,مش بيظهر دور الوزارة,0.615385,0.000000,0.615385,المكان ده هو السكشن. دور السكشن محوري. هو الجس...,المكان ده هو السكشن دور السكشن محوري هو الجسر ...
1492,wavs/sales_29_00021.wav,sales,0.135135,هل الشخص يشعر بأنه ليس شاركاً مثل الآخرين؟,مش شاطر زي ما الناس شايفاه.,مش بيعمل كويس زي ما الناس شايفةه.,لو الشغل ممكن يكون موجود حتى لو الشغل معقولة و...,0.615385,0.000000,0.615385,اللي هو الواحد يبقى حاسس إنه مش شاطر زي ما الن...,اللي هو الواحد يبقى حاسس إنه مش شاطر زي ما الن...
1461,wavs/sales_20_00009.wav,sales,0.081081,كم عدد الأفكار الممكن أن تكون حلًا للمشكلة؟,أفكار كتير ممكن تكون بتحل مشكلة.,كل الأفكار كتير ممكن تكون بنعكس مشكلة.,لو النقطة دي بالذات بتاعت ليها تشتري فلوس دي ج...,0.615385,0.000000,0.615385,بالذات دي حتة مهمة اوي. أيوة النطة دي بالذات ب...,بالذات دي حتة مهمة اوي. أيوة النقطة دي بالذات ...


## 15. Zip model and outputs for download

Run this at the end if you want one downloadable artifact.


In [ ]:
import shutil

# =====
# ARTIFACT_DIR = "arat5_niletts_final_outputs"
# Path(ARTIFACT_DIR).mkdir(exist_ok=True)

# files_to_copy = [
#     "niletts_qa_test_reference_with_asr_context.csv",
#     "niletts_arat5_llm_reference_eval_predictions.csv",
#     "niletts_arat5_llm_asr_eval_predictions.csv",
#     "niletts_arat5_reference_vs_asr_comparison.csv",
#     "niletts_arat5_reference_vs_asr_summary_metrics.json",
#     "niletts_arat5_domain_level_results.csv",
#     "niletts_arat5_worst_asr_error_examples.csv",
# ]

# for f in files_to_copy:
#     if Path(f).exists():
#         shutil.copy(f, Path(ARTIFACT_DIR) / f)

# if Path(BEST_MODEL_DIR).exists():
#     if Path(ARTIFACT_DIR, BEST_MODEL_DIR).exists():
#         shutil.rmtree(Path(ARTIFACT_DIR, BEST_MODEL_DIR))
#     shutil.copytree(BEST_MODEL_DIR, Path(ARTIFACT_DIR, BEST_MODEL_DIR))

# zip_path = shutil.make_archive(ARTIFACT_DIR, "zip", ARTIFACT_DIR)
zip_path = shutil.make_archive(base_name=str(ARTIFACT_DIR), format="zip", root_dir=str(ARTIFACT_DIR))
# =====

print("Created:", zip_path)

Created: /root/arat5_niletts_final_outputs.zip
